In [ ]:
# ── Step 1: Imports ──
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split

# ── Step 2: Dataset ──
torch.manual_seed(42)  # IMPORTANT — same seed as training

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset      = datasets.ImageFolder("dataset/", transform=transform)
train_size   = int(0.8 * len(dataset))
test_size    = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
test_loader  = DataLoader(test_dataset, batch_size=32)

# ── Step 3: MobileNetV2 ──
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mobilenet = models.mobilenet_v2(pretrained=True).to(device)
mobilenet.classifier = nn.Identity()
mobilenet.eval()

# ── Step 4: Extract test features ──
def feature_extraction(loader, model):
    all_features, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            all_features.append(model(images).cpu())
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

_, test_features, test_labels = None, *feature_extraction(test_loader, mobilenet)
X_test  = test_features.numpy()
y_test  = test_labels.numpy()
print("X_test shape:", X_test.shape)